In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Load data with features already extracted
df = pd.read_pickle('features_extracted.pkl')
df.head()

## Topic Modeling with Latent Dirichlet Allocation (LDA)

We use LDA to identify common lyrical themes in the dataset. We'll build a count matrix from our clean text and experiment with different numbers of topics.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Step 1 — Build the Corpus (Bag-of-Words)
# We use the clean text string (lyrics_preproc_v2)
cv = CountVectorizer(
    min_df=5,       # word must appear in at least 5 documents
    max_df=0.85,    # ignore words appearing in >85% of docs
    max_features=3000
)
count_matrix = cv.fit_transform(df["lyrics_preproc_v2"])
count_matrix.shape

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# Step 2 — Choose Number of Topics (k) - Perplexity check
# Note: running this may take a few seconds
perplexities = []
ks = range(4, 12)
for k in ks:
    lda = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=20)
    lda.fit(count_matrix)
    perplexities.append(lda.perplexity(count_matrix))

plt.plot(list(ks), perplexities, marker='o')
plt.xlabel("Number of Topics")
plt.ylabel("Perplexity (lower = better)")
plt.title("LDA Perplexity by Number of Topics")
plt.show()

Based on expected themes (e.g., love, street, party, sadness, etc.) and evaluating the perplexity, we fit the LDA model with $K=7$ topics.

In [ ]:
# Step 3 — Fit the LDA model
K = 7

lda_final = LatentDirichletAllocation(
    n_components=K,
    random_state=42,
    max_iter=50,
    learning_method="batch"
)
lda_final.fit(count_matrix)

# Step 4 — Inspect Topics (Print Top Words per Topic)
feature_names = cv.get_feature_names_out()

for i, topic in enumerate(lda_final.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-15:]][::-1]
    print(f"Topic {i}: ", ", ".join(top_words))

Now we extract the topic distributions as features for each song, finding the dominant topic and topic confidence.

In [ ]:
# Step 5 — Extract Per-Song Topic Distributions
topic_dist = lda_final.transform(count_matrix)  # shape: (790, K)

topic_cols = [f"topic_{i}" for i in range(K)]
topic_df = pd.DataFrame(topic_dist, columns=topic_cols, index=df.index)

# Drop existing topic columns if we are re-running
cols_to_drop = [c for c in df.columns if c in topic_cols or c in ["dominant_topic", "topic_confidence"]]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

# Add topic distributions to main dataframe
df = pd.concat([df, topic_df], axis=1)

# Step 6 — Derive a Summary Feature: Dominant Topic
df["dominant_topic"] = topic_dist.argmax(axis=1)
df["topic_confidence"] = topic_dist.max(axis=1)  # how "pure" the assignment is

df[["lyrics_preproc_v2", "dominant_topic", "topic_confidence"]].head()

### Visualizing Dominant Topics and Chart Persistence

In [ ]:
# Step 7 — Visualize
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# a) Topic distribution across the corpus
df["dominant_topic"].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title("Dominant Topic Distribution Across Songs")
axes[0].set_xlabel("Topic #")
axes[0].set_ylabel("Number of Songs")

# b) Topic vs. days_in_top50 (boxplot)
sns.boxplot(x="dominant_topic", y="days_in_top50", data=df, ax=axes[1])
axes[1].set_title("Days in Top 50 by Dominant Topic")
axes[1].set_xlabel("Topic #")
axes[1].set_ylabel("Days in Top 50")

plt.tight_layout()
plt.show()

### Top 5 Songs per Topic

Here we find the 5 songs most strongly associated with each topic (highest topic probability) and visualize them.

In [ ]:
fig, axes = plt.subplots(K, 1, figsize=(10, 4 * K), sharex=False)
plt.subplots_adjust(hspace=0.5)

for i in range(K):
    # Get top 5 songs for topic i with highest probability
    top_songs = df.sort_values(by=f"topic_{i}", ascending=False).head(5)
    
    # Combine song name and artist for better context in plots
    labels = top_songs["name"] + " - " + top_songs["artists"]
    
    ax = axes[i]
    sns.barplot(x=top_songs[f"topic_{i}"], y=labels, ax=ax, palette="viridis")
    
    ax.set_title(f"Top 5 Songs for Topic {i}")
    ax.set_xlabel("Topic Probability")
    ax.set_ylabel("")
    ax.set_xlim(0, 1) # Probability is between 0 and 1

plt.tight_layout()
plt.show()

### Advanced Topic Exploration

Let's dive a bit deeper into what these topics represent by exploring the actual word weights, and analyzing how our topics correlate with other features like sentiment and audio attributes.

In [ ]:
# 1. Word Importance per Topic (Top Words Bar Chart)
fig, axes = plt.subplots(int(np.ceil(K/2)), 2, figsize=(15, 4 * np.ceil(K/2)), sharex=False)
axes = axes.flatten()

for i, topic in enumerate(lda_final.components_):
    # Get top 10 words and their weights
    top_word_indices = topic.argsort()[-10:]
    top_words = [feature_names[j] for j in top_word_indices]
    top_weights = [topic[j] for j in top_word_indices]
    
    ax = axes[i]
    ax.barh(top_words, top_weights, color='teal')
    ax.set_title(f"Topic {i} Word Weights", fontsize=14)
    ax.set_xlabel("Weight")

# Hide empty subplots if K is odd
for i in range(K, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

#### Topics vs. Sentiment & Audio Features

Are certain lyrical topics inherently more "danceable" or more "sad" (low valence/negative sentiment)? Let's plot the correlations between a song's **Topic Probabilities** and its **Audio/Sentiment features**.

In [ ]:
# 2. Correlation Matrix: Topics vs Audio/Sentiment Features
features_to_correlate = ['danceability', 'energy', 'valence', 'tempo', 'sentiment_polarity', 'sentiment_intensity']

corr_df = df[topic_cols + features_to_correlate].corr()
# We only care about the correlation between Topics and the explicit features
topic_feature_corr = corr_df.loc[topic_cols, features_to_correlate]

plt.figure(figsize=(10, 6))
sns.heatmap(topic_feature_corr, annot=True, cmap='RdBu', center=0, fmt=".2f")
plt.title("Correlation between Topics and Continuous Features")
plt.ylabel("Topic")
plt.xlabel("Feature")
plt.show()

We can further plot how sentiment shifts alongside the dominant topics.

In [ ]:
# 3. Sentiment vs Dominant Topic
plt.figure(figsize=(10, 5))
sns.violinplot(x="dominant_topic", y="sentiment_polarity", data=df, palette="muted")
plt.title("Sentiment Polarity Distribution by Dominant Topic")
plt.xlabel("Topic #")
plt.ylabel("Sentiment Polarity (Compound Scale)")
plt.axhline(0, color='gray', linestyle='--') # neutral sentiment line
plt.show()

Finally, we save the resulting dataset enriched with text features and topic distributions.

In [ ]:
# Step 8 — Save the enriched DataFrame
df.to_pickle("lyrics_with_topics.pkl")
df.to_csv("lyrics_with_topics.csv", index=False)